# 🖍️ 18 · Teaching a network to annotate a cell: CNN segmentation of cellular tomograms
### Ten hand-labelled tiles per feature, one small network per feature, slice by slice through the whole tomogram

**BiologyToResearch** · Part D — Deep learning on subtomograms (2017–2024) · Module 18 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/18_cnn_annotation_cellular_tomograms.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 2–3 min to run everything); a GPU is optional |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, PyTorch |
| **Prerequisites** | [05 · Deep learning](05_deep_learning_pytorch.ipynb) (convolutions, training loops); [07 · Tomography](07_tomography_missing_wedge.ipynb) and [09 · In silico cell](09_in_silico_cell_simulation.ipynb) (simulated tomograms); [17 · Segmentation](17_interpretability_segmentation.ipynb) (voxel-wise labels, Dice) |
| **Estimated time** | 100 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Chen, Dai, Sun, Jonasch, He, Schmid, Chiu & Ludtke 2017 (CNNs for automated annotation of cellular cryo-electron tomograms, *Nat Methods*) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments are small, controlled *toy versions* of published methods (a 64 × 192 × 192-voxel simulated tomogram,
> networks with 12 instead of 40 channels, two minutes of training). They illustrate the core ideas; they do **not**
> reproduce the paper's numbers. **Nothing is stored in the repository:** molecules are real Protein Data Bank (PDB)
> models, downloaded the first time a notebook needs them and cached on your computer (offline, code-built stand-ins
> of the same size and symmetry are used instead); all images and data are simulated by code. Our "hand-drawn"
> annotations are perfect simulation labels, which real ones never are.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why manual annotation of tomograms is a bottleneck and how a convolutional network trained on a handful of
   labelled tiles removes it.
2. **Simulate** a cellular tomogram with membranes, microtubules, ribosomes and gold fiducials, and **preprocess** it as
   the paper does (band-pass, normalise, clamp).
3. **Build** the paper's small network — large 15 × 15 kernels, one pooling step, a capped linear output — and **train**
   one network per feature from ~10 positive and ~100 negative 64 × 64 tiles.
4. **Apply** the networks slice by slice to the whole tomogram, **merge** several feature maps into one annotation, and
   **measure** precision, recall and Dice.
5. **Test** what the paper warns about: features nobody trained for, other tomograms, other imaging conditions.
6. **Turn** an annotation into particle positions and **relate** it to subtomogram averaging.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | tools, PyTorch | 5 min |
| 1 · A cell to annotate | membranes, microtubules, ribosomes, gold; preprocessing | 15 min |
| 2 · Ten labels per feature | positive and negative tiles, rotation augmentation | 10 min |
| 3 · The network | 15 × 15 kernels, pooling, min(1, x), log loss | 15 min |
| 4 · Train one network per feature | three trainings | 10 min |
| 5 · Annotate the whole cell | slice by slice, merge, evaluate | 15 min |
| 6 · Where it goes wrong | gold beads, new tomograms, new imaging conditions | 15 min |
| 7 · From annotation to particles | ribosome positions | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A tomogram of a neuron-like PC12 cell or a trypanosome holds membranes, mitochondria, vesicles, microtubules, actin,
ribosomes — the cell's whole architecture in 3D. To *use* it, someone must mark what is where: **annotation** (or
segmentation). By hand, slice by slice, the authors note this can take "as much as one man week" for one large tomogram.
In 2017 Muyuan Chen, Steven Ludtke, Wah Chiu and colleagues showed that a small convolutional neural network, trained on
a few tiles that a person labels in minutes, could annotate whole tomograms automatically — one of the first deep-learning
tools in cellular cryo-ET, distributed in EMAN2. Their annotations fed straight into subtomogram averages of
microtubules and ribosomes.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook runs on a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.colors import ListedColormap
from scipy import ndimage

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
T_START = time.time()
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

Helpers copied from [`course_toolkit.py`](../course_toolkit.py): the ribosome of the molecular zoo (Module 00),
rotations (Module 02), `simulate_tomogram` (CTF, missing wedge and noise — Modules 06, 07, 09) and `match_points`
(Module 11).

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_EXTRA_ENTRIES = {"5a1a": dict(name="β-galactosidase", organism="E. coli", mass_kDa=465, symmetry="D2")}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
def standin_atoms(pid):
    """Atomic model (dict of arrays) of the code-built stand-in for zoo entry `pid` (Å, centred at the origin).

    Used when the real entry cannot be downloaded: ideal α-helices arranged with the size, mass and symmetry of the
    PDB entry. Deterministic (the random numbers are seeded by the entry's name)."""
    pid = pid.lower()
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    return atoms


def pdb_cache_dir():
    """Folder (outside the repository) where downloaded PDB files are kept."""
    return os.environ.get("BIO2RESEARCH_PDB_CACHE") or os.path.join(os.path.expanduser("~"), ".cache",
                                                                    "biology_to_research", "pdb")


def _pdb_urls(pid, kind):
    """Download addresses of one PDB file: kind = 'entry' or 'assembly1' (mmCIF) or 'pdb' (legacy PDB format)."""
    pid = pid.lower()
    mid = pid[1:3]
    if kind == "entry":
        return [f"https://files.rcsb.org/download/{pid.upper()}.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/mmCIF/{mid}/{pid}.cif.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/{pid}.cif"]
    if kind == "assembly1":
        return [f"https://files.rcsb.org/download/{pid.upper()}-assembly1.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/assemblies/mmCIF/divided/{mid}/{pid}-assembly1.cif.gz"]
    if kind == "pdb":
        return [f"https://files.rcsb.org/download/{pid.upper()}.pdb.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/pdb/{mid}/pdb{pid}.ent.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{pid}.ent"]
    raise ValueError(kind)


def fetch_pdb(pid, kind="entry", timeout=60):
    """Text of a PDB file, downloaded once and cached outside the repository. Returns (text, where_from), or
    (None, reason) when no mirror can be reached (or BIO2RESEARCH_STANDINS is set)."""
    import gzip
    import urllib.request
    name = {"entry": f"{pid.lower()}.cif.gz", "assembly1": f"{pid.lower()}-assembly1.cif.gz",
            "pdb": f"{pid.lower()}.pdb.gz"}[kind]
    path = os.path.join(pdb_cache_dir(), name)
    if os.path.exists(path):
        return gzip.decompress(open(path, "rb").read()).decode("utf-8", "replace"), f"cached copy ({path})"
    if os.environ.get("BIO2RESEARCH_STANDINS"):
        return None, "BIO2RESEARCH_STANDINS is set"
    reason = "no mirror tried"
    for url in _pdb_urls(pid, kind):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                raw = response.read()
            text = (gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw).decode("utf-8", "replace")
            if ("_atom_site." not in text) and ("\nATOM  " not in text):
                raise ValueError("no atoms in the downloaded file")
            os.makedirs(pdb_cache_dir(), exist_ok=True)
            with open(path, "wb") as f:
                f.write(gzip.compress(text.encode()))
            return text, url
        except Exception as e:  # noqa: BLE001 — any failure: try the next mirror
            reason = f"{type(e).__name__}: {str(e)[:120]}"
    return None, reason


def _cif_tokens(line):
    """Split one line of an mmCIF loop into values (quoted values may contain spaces)."""
    if "'" not in line and '"' not in line:
        return line.split()
    import re
    toks = re.findall(r"'(?:[^']|'(?=\S))*'|\"(?:[^\"]|\"(?=\S))*\"|\S+", line)
    return [t[1:-1] if len(t) > 1 and t[0] == t[-1] and t[0] in "'\"" else t for t in toks]


_WATER = {"HOH", "DOD", "WAT", "H2O"}
def read_mmcif_atoms(text):
    """Atoms of the first model in an mmCIF file → dict(xyz, name, element, resname, chain, resnum).

    Reads the `_atom_site` loop; skips water and hydrogen, and keeps only the first of alternative atom locations."""
    lines = text.splitlines()
    i = next(k for k, line in enumerate(lines) if line.startswith("_atom_site."))
    cols = []
    while i < len(lines) and lines[i].startswith("_atom_site."):
        cols.append(lines[i].split()[0][len("_atom_site."):])
        i += 1
    c = {name: k for k, name in enumerate(cols)}
    get = lambda *names: next((c[n] for n in names if n in c), None)   # noqa: E731
    ix, iy, iz = c["Cartn_x"], c["Cartn_y"], c["Cartn_z"]
    i_el, i_name = get("type_symbol"), get("auth_atom_id", "label_atom_id")
    i_res, i_ch = get("auth_comp_id", "label_comp_id"), get("auth_asym_id", "label_asym_id")
    i_num, i_alt, i_mod = get("auth_seq_id", "label_seq_id"), get("label_alt_id"), get("pdbx_PDB_model_num")
    xyz, name, element, resname, chain, resnum = [], [], [], [], [], []
    first_model, buf = None, []
    for line in lines[i:]:
        if not line or line.startswith(("#", "_", "loop_", "data_")):
            break
        buf += _cif_tokens(line)
        while len(buf) >= len(cols):
            row, buf = buf[:len(cols)], buf[len(cols):]
            if i_mod is not None:
                first_model = row[i_mod] if first_model is None else first_model
                if row[i_mod] != first_model:
                    continue
            el = row[i_el].upper() if i_el is not None else row[i_name][:1].upper()
            if el in ("H", "D") or row[i_res] in _WATER:
                continue
            if i_alt is not None and row[i_alt] not in (".", "?", "A", "1"):
                continue
            xyz.append((float(row[ix]), float(row[iy]), float(row[iz])))
            name.append(row[i_name])
            element.append(el.capitalize() if len(el) > 1 else el)
            resname.append(row[i_res])
            chain.append(row[i_ch])
            num = row[i_num] if i_num is not None else "0"
            resnum.append(int(num) if num.lstrip("-").isdigit() else 0)
    return dict(xyz=np.array(xyz, np.float32), name=np.array(name), element=np.array(element),
                resname=np.array(resname), chain=np.array(chain), resnum=np.array(resnum))


_ATOMIC_MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "P": 30.974, "S": 32.06, "Se": 78.97, "Mg": 24.305,
                "Zn": 65.38, "Fe": 55.845, "Mn": 54.938, "K": 39.098, "Na": 22.99, "Cl": 35.45, "Ca": 40.078}
def model_mass_kDa(atoms):
    """Mass of an atomic model without hydrogens, plus ~8 % for the missing hydrogens (kDa)."""
    return 1.08 * sum(_ATOMIC_MASS.get(e, 12.0) for e in atoms["element"]) / 1000.0


def canonical_frame(atoms):
    """Centre a model and rotate it onto its principal axes — longest extent along x, shortest along z — with the
    signs fixed by the skewness of the atom distribution. A proteasome barrel then lies along x and a ring
    (RuBisCO, p97) lies in the x–y plane, like the stand-ins."""
    xyz = atoms["xyz"].astype(np.float64)
    X = xyz - xyz.mean(0)
    evecs = np.linalg.eigh(np.cov(X.T))[1][:, ::-1]                 # columns: principal axes, largest first
    Y = X @ evecs
    sign = np.sign(np.sum(Y ** 3, axis=0))
    sign[sign == 0] = 1.0
    Y *= sign
    if np.linalg.det(evecs * sign) < 0:                             # keep a proper rotation (no mirror image)
        Y[:, 2] *= -1
    out = dict(atoms)
    out["xyz"] = Y.astype(np.float32)
    return out


def pdb_atoms(pid, verbose=False):
    """Real atomic model of a PDB entry, in the canonical frame. Returns (atoms, description), or (None, reason).

    Uses the deposited entry unless its mass is far from ZOO's reference mass, in which case the first biological
    assembly is tried and the closer of the two kept (e.g. one ribosome rather than the two of a crystal)."""
    pid = pid.lower()
    ref = ZOO.get(pid, {}).get("mass_kDa") or _EXTRA_ENTRIES.get(pid, {}).get("mass_kDa")
    candidates, reason = [], ""
    for kind in ("entry", "assembly1"):
        if candidates and (ref is None or abs(np.log(model_mass_kDa(candidates[0][0]) / ref)) < np.log(1.3)):
            break
        text, where = fetch_pdb(pid, kind)
        if text is None:
            reason = where
            continue
        try:
            atoms = read_mmcif_atoms(text)
        except (StopIteration, KeyError, ValueError) as e:
            reason = f"could not read the file ({type(e).__name__})"
            continue
        if len(atoms["xyz"]):
            label = "deposited entry" if kind == "entry" else "biological assembly 1"
            candidates.append((atoms, f"PDB {pid.upper()} ({label}; {len(atoms['xyz']):,} atoms; from {where})"))
    if not candidates:
        return None, reason
    atoms, desc = min(candidates, key=lambda a: abs(np.log(model_mass_kDa(a[0]) / ref)) if ref else 0.0)
    if verbose:
        print(desc)
    return canonical_frame(atoms), desc


_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays, Å, centred) of zoo entry `pid`: the real PDB model when it can be downloaded, the
    code-built stand-in otherwise. The source is recorded in `atoms_source(pid)`."""
    pid = pid.lower()
    if pid not in _ATOMS_CACHE:
        atoms, desc = pdb_atoms(pid)
        if atoms is None:
            atoms, desc = standin_atoms(pid), f"code-built stand-in modelled on PDB {pid.upper()} (download failed: {desc})"
        _ATOMS_CACHE[pid] = (atoms, desc)
    return _ATOMS_CACHE[pid][0]


def atoms_source(pid):
    """Where the atoms of zoo entry `pid` came from (a real PDB file or the offline stand-in)."""
    zoo_atoms(pid)
    return _ATOMS_CACHE[pid.lower()][1]


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0, "Mg": 12.0, "Zn": 30.0, "Fe": 26.0, "K": 19.0,
      "Na": 11.0, "Cl": 17.0, "Ca": 20.0, "Mn": 25.0, "Se": 34.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of a zoo entry, painted from its atoms (`zoo_atoms`)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, atoms_source(pid)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def electron_wavelength_A(kV: float = 300.0) -> float:
    """Relativistic electron wavelength in Å (0.0197 Å at 300 kV)."""
    V = kV * 1e3
    return 12.2643 / np.sqrt(V * (1 + 0.978466e-6 * V))


def ctf_2d(shape, pixel_A: float, defocus_um: float = 3.0, kV: float = 300.0,
           Cs_mm: float = 2.7, amp_contrast: float = 0.07, B: float = 0.0) -> np.ndarray:
    """Contrast transfer function on an unshifted FFT grid (underfocus is positive).

    CTF(k) = −[√(1−A²)·sin χ(k) + A·cos χ(k)]·exp(−B k²/4),
    χ(k) = π λ Δf k² − (π/2) C_s λ³ k⁴, with k in 1/Å.
    """
    lam = electron_wavelength_A(kV)
    k2 = sum(kk**2 for kk in freq_grid(shape)) / pixel_A**2
    dz = defocus_um * 1e4
    cs = Cs_mm * 1e7
    chi = np.pi * lam * dz * k2 - 0.5 * np.pi * cs * lam**3 * k2**2
    ctf = -(np.sqrt(1 - amp_contrast**2) * np.sin(chi) + amp_contrast * np.cos(chi))
    return (ctf * np.exp(-B * k2 / 4)).astype(np.float32)


def match_points(pred, true, radius: float):
    """Greedy one-to-one matching of predicted to true coordinates within ``radius``.

    Returns dict(precision, recall, f1, tp, fp, fn).
    """
    true = np.asarray(true, float)
    pred = np.asarray(pred, float).reshape(-1, true.shape[-1] if true.ndim == 2 else 3)
    if len(pred) == 0 or len(true) == 0:
        tp = 0
    else:
        d = np.linalg.norm(pred[:, None, :] - true[None, :, :], axis=-1)
        used_p, used_t, tp = set(), set(), 0
        for i, j in zip(*np.unravel_index(np.argsort(d, axis=None), d.shape)):
            if d[i, j] > radius:
                break
            if i not in used_p and j not in used_t:
                used_p.add(i); used_t.add(j); tp += 1
    fp, fn = len(pred) - tp, len(true) - tp
    p = tp / max(len(pred), 1)
    r = tp / max(len(true), 1)
    return dict(precision=p, recall=r, f1=2 * p * r / max(p + r, 1e-12), tp=tp, fp=fp, fn=fn)


def simulate_tomogram(clean: np.ndarray, rng, snr: float = 0.05, tilt_max_deg: float = 60.0,
                      ctf_pixel_A: float | None = 10.0, defocus_um: float = 5.0) -> np.ndarray:
    """Fast tomogram simulation: (optional 3D CTF-like filter) → missing wedge → noise.

    The CTF is applied isotropically here (a simplification of per-tilt CTF).
    """
    v = clean
    if ctf_pixel_A:
        ctf = ctf_2d(clean.shape, ctf_pixel_A, defocus_um=defocus_um, kV=300, B=0)
        v = apply_fourier_mask(v, -ctf)          # sign flip: dense objects appear bright
    v = apply_fourier_mask(v, missing_wedge_mask(v.shape, tilt_max_deg))
    return add_noise_snr(v, snr, rng)
print("🧰 toolkit loaded")

---
## 1 · A cell to annotate

> 📍 **Where we are.** Modules 16–17 classified and segmented *subtomograms* — small boxes around particles found
> earlier. This module goes one level up: annotate the **whole tomogram** — membranes, filaments and particles alike —
> with networks trained from a few minutes of labelling. Module 32 returns to this task with today's foundation models.

We build a 64 × 192 × 192-voxel piece of cell at 1 nm per voxel with four kinds of objects, and keep a label for every
voxel: **membranes** (vesicles, each a lipid bilayer of two leaflets 4 nm apart), **microtubules** (hollow tubes 25 nm
wide, seen as pairs of parallel lines in a slice), **ribosomes** (the zoo's 70S ribosome, PDB 4V4R, in random orientations) and a
few **gold beads** — dense nanoparticles that labs add as fiducial markers (Module 07) and that we will *not* train on.

In [ ]:
# ── 1.1 Build a labelled piece of cell ──
SHAPE = (64, 192, 192)
LABEL_NAMES = ["background", "membrane", "microtubule", "ribosome", "gold"]
RIBO_TEMPLATE = prepare_map(zoo_map("4v4r")[0], size=32, sigma=1.0)


def make_cell(rng, n_ribosomes: int = 40, n_vesicles: int = 3, n_microtubules: int = 2, n_gold: int = 6):
    """Clean density, voxel labels (0–4) and ribosome centres (z, y, x) of a simulated piece of cell."""
    vol, labels = np.zeros(SHAPE, np.float32), np.zeros(SHAPE, np.int8)
    zz, yy, xx = np.meshgrid(*[np.arange(s, dtype=np.float32) for s in SHAPE], indexing="ij")
    for _ in range(n_vesicles):                                             # bilayer: two leaflets 4 nm apart
        c = np.array([rng.uniform(20, 44), rng.uniform(30, 162), rng.uniform(30, 162)]); r0 = rng.uniform(14, 24)
        r = np.sqrt((zz - c[0]) ** 2 + (yy - c[1]) ** 2 + (xx - c[2]) ** 2)
        vol += 0.9 * (np.exp(-(r - r0 + 2) ** 2 / 2) + np.exp(-(r - r0 - 2) ** 2 / 2))
        labels[np.abs(r - r0) < 3.5] = 1
    for _ in range(n_microtubules):                                         # hollow tube, outer diameter ≈ 25 nm
        a = rng.uniform(0, np.pi); d = np.array([0, np.sin(a), np.cos(a)])
        p0 = np.array([rng.uniform(22, 42), rng.uniform(40, 150), rng.uniform(40, 150)])
        rel = np.stack([zz - p0[0], yy - p0[1], xx - p0[2]], -1)
        perp = np.linalg.norm(rel - (rel @ d)[..., None] * d, axis=-1)
        vol += np.exp(-(perp - 10.5) ** 2 / (2 * 1.6 ** 2))
        labels[(np.abs(perp - 10.5) < 3.0) & (labels == 0)] = 2
    centres = []
    for _ in range(n_ribosomes * 100):                                       # ribosomes, no overlaps
        if len(centres) >= n_ribosomes:
            break
        c = np.array([rng.uniform(16, 48), rng.uniform(16, 176), rng.uniform(16, 176)])
        if any(np.linalg.norm(c - q) < 26 for q in centres):
            continue
        t = rotate_volume(RIBO_TEMPLATE, random_rotation(rng))
        z0, y0, x0 = np.round(c).astype(int) - 16
        box = (slice(z0, z0 + 32), slice(y0, y0 + 32), slice(x0, x0 + 32))
        if (labels[box][t > 0.15] != 0).any():
            continue
        vol[box] += 1.2 * t
        labels[box][t > 0.25] = 3
        centres.append(c)
    for _ in range(n_gold):                                                 # gold beads: small and very dense
        c = np.array([rng.uniform(10, 54), rng.uniform(10, 182), rng.uniform(10, 182)])
        r = np.sqrt((zz - c[0]) ** 2 + (yy - c[1]) ** 2 + (xx - c[2]) ** 2)
        vol += 4.0 * np.exp(-r ** 2 / (2 * 2.5 ** 2))
        labels[(r < 4) & (labels == 0)] = 4
    return vol, labels, np.array(centres)


t0 = time.time()
clean, labels, ribo_centres = make_cell(rng)
print(f"cell built in {time.time() - t0:.1f} s; {len(ribo_centres)} ribosomes; voxel fractions: "
      + ", ".join(f"{n} {np.mean(labels == k):.1%}" for k, n in enumerate(LABEL_NAMES)))

### 1.2 Image it and preprocess it as the paper does

The toolkit's `simulate_tomogram` adds a CTF (4 µm defocus), the ±60° missing wedge and noise (SNR 0.1). Chen *et al.*
then **band-pass** the tomogram (low-pass at ~20 Å to suppress noise, high-pass at ~4000 Å to remove slow background
variations), **normalise** it to mean 0 and standard deviation 1, and **clamp** values beyond ±3 standard deviations, so
that one very dense object (a gold bead!) cannot dominate.

In [ ]:
# ── 1.2 Simulate the tomogram and preprocess it ──
def preprocess(tomo: np.ndarray) -> np.ndarray:
    """Band-pass (≈ 20 Å low-pass, slow-background removal), normalise, clamp at ±3 sd."""
    v = ndimage.gaussian_filter(tomo, 1.0) - ndimage.gaussian_filter(tomo, 30.0)
    v = (v - v.mean()) / v.std()
    return np.clip(v, -3, 3).astype(np.float32)


tomogram = preprocess(simulate_tomogram(clean, rng, snr=0.1, tilt_max_deg=60, ctf_pixel_A=10.0, defocus_um=4.0))
CMAP = ListedColormap(["#00000000", "tab:orange", "tab:green", "tab:red", "tab:purple"])
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
axes[0].imshow(clean[32]); axes[0].set_title("clean density (z = 32)", fontsize=9)
axes[1].imshow(tomogram[32]); axes[1].set_title("simulated, preprocessed tomogram", fontsize=9)
axes[2].imshow(tomogram[32]); axes[2].imshow(labels[32], cmap=CMAP, vmin=0, vmax=4, alpha=0.6)
axes[2].set_title("labels: membrane, microtubule, ribosome, gold", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

---
## 2 · Ten labels per feature

The paper's recipe for each feature: a person picks about **10 positive tiles** of 64 × 64 pixels from slices of the
tomogram and draws the feature on them, plus about **100 negative tiles** that do not contain it. To balance the two, the
positive tiles are **replicated in random orientations**. One network is trained per feature (the authors report e.g. 10
positive / 55 negative tiles for membranes and 10 / 86 for microtubules in a PC12 cell, 5 / 125 for ribosomes in a
trypanosome).

⚠️ **A trap worth knowing.** Rotating an image by an arbitrary angle interpolates between pixels, which slightly
*smooths* the noise. If only the positive tiles were rotated, the network could learn the shortcut "smooth noise ⇒
feature" instead of the feature itself. We therefore give **every** tile, positive and negative, a fresh random
orientation in every epoch (and replicate the positives 10×).

In [ ]:
# ── 2.1 Pick positive and negative tiles for a feature ──
def pick_tiles(tomo, labs, feature: int, rng, n_pos: int = 10, n_neg: int = 100, min_fraction: float = 0.05):
    """Random 64 × 64 slice tiles: positives contain ≥ min_fraction of the feature, negatives none of it."""
    pos, neg = [], []
    for _ in range(20000):
        if len(pos) >= n_pos and len(neg) >= n_neg:
            break
        z, (y, x) = rng.integers(8, SHAPE[0] - 8), rng.integers(0, SHAPE[1] - 64, 2)
        tile, mask = tomo[z, y:y + 64, x:x + 64], (labs[z, y:y + 64, x:x + 64] == feature)
        if mask.mean() >= min_fraction and len(pos) < n_pos:
            pos.append((tile, mask))
        elif not mask.any() and len(neg) < n_neg:
            neg.append((tile, mask))
    return pos, neg


def random_orientation(tile, mask, rng):
    """Rotate by a random angle (linear interpolation, mirrored borders) and maybe flip."""
    angle = rng.uniform(0, 360)
    tile = ndimage.rotate(tile, angle, reshape=False, order=1, mode="reflect")
    mask = ndimage.rotate(mask.astype(np.float32), angle, reshape=False, order=1, mode="reflect") > 0.5
    if rng.random() < 0.5:
        tile, mask = tile[:, ::-1], mask[:, ::-1]
    return np.ascontiguousarray(tile), np.ascontiguousarray(mask)


tiles = {f: pick_tiles(tomogram, labels, f, np.random.default_rng(f)) for f in (1, 2, 3)}
fig, axes = plt.subplots(3, 6, figsize=(10, 5.4))
for row, f in enumerate((1, 2, 3)):
    pos, neg = tiles[f]
    for j in range(4):
        axes[row, j].imshow(pos[j][0]); axes[row, j].imshow(pos[j][1], cmap=ListedColormap(["#00000000", "tab:red"]), alpha=0.5)
    for j in range(4, 6):
        axes[row, j].imshow(neg[j][0])
    axes[row, 0].set_ylabel(LABEL_NAMES[f], fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_title("positive tiles (red = label)", fontsize=8, loc="left"); axes[0, 4].set_title("negative tiles", fontsize=8, loc="left")
plt.tight_layout(); plt.show()

---
## 3 · The network

The 2017 network is small and unusual by today's standards — worth reading layer by layer:

| layer | what it does | paper | ours |
|---|---|---|---|
| 1 · convolution + ReLU | 15 × 15 kernels on the 64 × 64 tile | 40 kernels | 12 |
| 2 · max-pooling | 2 × 2 → 32 × 32 | same | same |
| 3 · "fully connected" layer | each of 40 neurons connects to every map of layer 1 with its own 15 × 15 kernel (40 × 40 = 1,600 kernels) — a convolution | 40 | 12 |
| 4 · output neuron | sums the maps plus an offset; activation $y=\min(1, x)$ | same | same |

*In words:* the large 15 × 15 kernels see 15 pixels at once — wide enough to recognise a membrane's double line or a
microtubule's pair of walls; pooling halves the image (so the target annotation is shrunk 2× too); the output is capped
at 1 but not below, a linear "yes, up to fully". The loss is the **logarithm of the squared error** between the output and
the shrunk annotation, with a small **L1 penalty** on the weights. (The paper trains with plain stochastic gradient
descent — batches of 20, 20 iterations, learning rate 0.01 falling 10 % per iteration; with our smaller network we use
Adam, which is more forgiving on a CPU.)

In [ ]:
# ── 3.1 The annotation network of Chen et al. 2017 (scaled down) ──
CHANNELS, KERNEL = 12, 15


class AnnotationCNN(nn.Module):
    """conv 15×15 (+ReLU) → 2×2 max-pool → conv 15×15 (+ReLU) → 1×1 sum + offset → min(1, x)."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, CHANNELS, KERNEL, padding=KERNEL // 2)
        self.conv2 = nn.Conv2d(CHANNELS, CHANNELS, KERNEL, padding=KERNEL // 2)
        self.out = nn.Conv2d(CHANNELS, 1, 1)

    def forward(self, x):
        x = F.max_pool2d(torch.relu(self.conv1(x)), 2)
        x = torch.relu(self.conv2(x))
        return torch.clamp(self.out(x), max=1.0)[:, 0]


net = AnnotationCNN()
print(f"parameters: {sum(p.numel() for p in net.parameters()):,}; output of a 64 × 64 tile: "
      f"{tuple(net(torch.zeros(1, 1, 64, 64)).shape[1:])}")

---
## 4 · Train one network per feature

In [ ]:
# ── 4.1 Training loop and three trainings ──
def train_feature(feature: int, epochs: int = 20, replicate: int = 10, seed: int = SEED):
    """Train an AnnotationCNN on the positive (replicated, randomly oriented) and negative tiles of one feature."""
    torch.manual_seed(seed)
    r = np.random.default_rng(seed)
    pos, neg = tiles[feature]
    model = AnnotationCNN()
    opt = torch.optim.Adam(model.parameters(), lr=5e-4)
    t0 = time.time()
    for ep in range(epochs):
        data = [random_orientation(t, m, r) for t, m in pos for _ in range(replicate)] + \
               [random_orientation(t, m, r) for t, m in neg]
        order = r.permutation(len(data))
        for i in range(0, len(data), 20):
            batch = [data[k] for k in order[i:i + 20]]
            x = torch.tensor(np.stack([b[0] for b in batch])[:, None])
            y = torch.tensor(np.stack([b[1].reshape(32, 2, 32, 2).mean((1, 3)) for b in batch]), dtype=torch.float32)
            loss = torch.log(((model(x) - y) ** 2).mean() + 1e-6) + 1e-5 * sum(p.abs().sum() for p in model.parameters())
            opt.zero_grad(); loss.backward(); opt.step()
    print(f"{LABEL_NAMES[feature]:12s}: {len(pos)} positive × {replicate} orientations + {len(neg)} negative tiles, "
          f"{epochs} epochs in {time.time() - t0:.0f} s")
    return model


models = {f: train_feature(f) for f in (1, 2, 3)}

---
## 5 · Annotate the whole cell

### 5.1 Slice by slice

The trained networks are **convolutional**, so they accept a whole 192 × 192 slice, not only 64 × 64 tiles. The paper
applies them in **2D, slice by slice** through the tomogram — simpler networks, and the missing-wedge elongation along
z (Module 07) does not confuse them. The output is at half resolution (one value per 2 × 2 voxels).

In [ ]:
# ── 5.1 Apply each network to every slice ──
def annotate(model, tomo):
    """Feature map (z, y/2, x/2) from applying a 2D network to every z-slice."""
    with torch.no_grad():
        return model(torch.tensor(tomo[:, None])).numpy()


t0 = time.time()
feature_maps = {f: annotate(models[f], tomogram) for f in models}
labels_half = labels[:, ::2, ::2]                                     # ground truth at the output's resolution
print(f"3 × {SHAPE[0]} slices annotated in {time.time() - t0:.1f} s")


def scores(pred: np.ndarray, truth: np.ndarray) -> tuple:
    """Precision, recall and Dice of two boolean arrays."""
    tp = (pred & truth).sum()
    return tp / max(pred.sum(), 1), tp / max(truth.sum(), 1), 2 * tp / max(pred.sum() + truth.sum(), 1)


for f in models:
    own, bg = feature_maps[f][labels_half == f].mean(), feature_maps[f][labels_half == 0].mean()
    print(f"{LABEL_NAMES[f]:12s} network: mean output {own:.2f} on its feature, {bg:.2f} on background")

### 5.2 Merging the networks into one annotation

Each network answers its own yes/no question; a voxel may get a "yes" from two of them. The paper merges them by
**normalising** each network's output so that its mean over its own annotated positives is 1, then giving each voxel to
the network with the **highest** normalised value, with a threshold below which a voxel stays background.

In [ ]:
# ── 5.2 Normalise, take the strongest network, threshold ──
norm = {}
for f in models:
    pos, _ = tiles[f]
    with torch.no_grad():
        out_pos = models[f](torch.tensor(np.stack([t for t, _ in pos])[:, None])).numpy()
    lab_pos = np.stack([m.reshape(32, 2, 32, 2).mean((1, 3)) > 0.5 for _, m in pos])
    norm[f] = out_pos[lab_pos].mean()                                       # mean output on annotated positives
stack = np.stack([feature_maps[f] / norm[f] for f in (1, 2, 3)])
THRESHOLD = 0.6
merged = np.where(stack.max(0) > THRESHOLD, stack.argmax(0) + 1, 0)
print("normalisation factors (mean output on positives):", {LABEL_NAMES[f]: round(float(v), 2) for f, v in norm.items()})
for f in (1, 2, 3):
    p, r, d = scores(merged == f, labels_half == f)
    print(f"merged, {LABEL_NAMES[f]:12s}: precision {p:.2f}, recall {r:.2f}, Dice {d:.2f}")

fig, axes = plt.subplots(2, 3, figsize=(11, 6), gridspec_kw={"height_ratios": [3, 1.3]})
z, y = 32, 48                                                          # the xy slice and the xz plane we show
for col, (lab, title) in enumerate([(None, "tomogram"), (merged, "CNN annotation (merged)"), (labels_half, "truth")]):
    axes[0, col].imshow(tomogram[z, ::2, ::2])
    axes[1, col].imshow(tomogram[:, 2 * y, ::2], aspect=0.5)           # z voxels are 1 nm, x voxels here 2 nm
    if lab is not None:
        axes[0, col].imshow(lab[z], cmap=CMAP, vmin=0, vmax=4, alpha=0.6)
        axes[1, col].imshow(lab[:, y], cmap=CMAP, vmin=0, vmax=4, alpha=0.6, aspect=0.5)
    axes[0, col].set_title(f"{title}, slice z = {z}", fontsize=9); axes[1, col].set_title("side view (xz)", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()
for f in (2, 3):                                                       # how tall (along z) are annotation and truth?
    h_true, h_pred = (labels_half == f).sum(0), (merged == f).sum(0)
    both = (h_true > 0) & (h_pred > 0)
    print(f"{LABEL_NAMES[f]:12s}: mean height along z of annotated columns {h_pred[both].mean():.1f} nm, true {h_true[both].mean():.1f} nm")

🔍 **What just happened?** Three networks, each trained on ten labelled tiles in well under a minute, annotated
membranes, microtubules and ribosomes through the whole volume. Errors concentrate at object borders (the annotation is
a fuzzy band, not a line), where two objects touch, and where noise mimics a feature. The **side view (xz)** shows a
limit no network can beat: the missing wedge (Module 07) erases surfaces that lie parallel to the slices. The top and
bottom walls of each microtubule are gone from the tomogram, which shows only the two side walls, like brackets ( ) —
so the annotation shows only those too (in the xy slice, the broad green band where the slice grazes a tube wall is
missed for the same reason). That is why microtubules have the lowest recall and come out *shorter* along z than they
are, while compact ribosomes come out slightly *taller* (the elongation Module 07 measured).

✏️ **Your turn.** Change `THRESHOLD` to `0.3` and `0.9`. Which feature gains recall, which loses precision?

---
## 6 · Where it goes wrong

The authors list the method's limits plainly: features the networks were not trained on give **unpredictable**
responses (carbon edges and gold beads produce false positives); networks see at most about 30 pixels at once, so very
large objects are recognised only by their edges; and a network works best on the cell type and imaging conditions it
was trained on. Let us test all three.

In [ ]:
# ── 6.1 Gold beads: a feature nobody trained for ──
gold = labels_half == 4
for f in models:
    print(f"{LABEL_NAMES[f]:12s} network: mean normalised output on gold beads {np.mean(feature_maps[f][gold]) / norm[f]:.2f} "
          f"(on its own feature {np.mean(feature_maps[f][labels_half == f]) / norm[f]:.2f}, "
          f"on background {np.mean(feature_maps[f][labels_half == 0]) / norm[f]:.2f})")
print(f"gold voxels annotated as something: {np.mean(merged[gold] > 0):.0%}  → as {np.bincount(merged[gold], minlength=4)[1:]} "
      f"(membrane, microtubule, ribosome)")

🔍 The networks never saw a gold bead, and at least one of them (here mostly the microtubule network) answers
"yes" to it — a dense dot after clamping looks enough like something it knows, and a network has no way to say "none
of the above". The cure in practice is to train an extra network for the nuisance feature (a
"gold" or "carbon" class) so that the merge step can claim those voxels for it.

In [ ]:
# ── 6.2 A new tomogram, and a new imaging condition ──
clean_b, labels_b, centres_b = make_cell(np.random.default_rng(101))
conditions = {"same conditions": dict(snr=0.1, defocus_um=4.0),
              "noisier, more defocus": dict(snr=0.04, defocus_um=6.0)}
for name, cond in conditions.items():
    tomo_b = preprocess(simulate_tomogram(clean_b, np.random.default_rng(7), tilt_max_deg=60, ctf_pixel_A=10.0, **cond))
    st = np.stack([annotate(models[f], tomo_b) / norm[f] for f in (1, 2, 3)])
    mg = np.where(st.max(0) > THRESHOLD, st.argmax(0) + 1, 0)
    dices = [scores(mg == f, labels_b[:, ::2, ::2] == f)[2] for f in (1, 2, 3)]
    print(f"new tomogram, {name:22s}: Dice membrane {dices[0]:.2f}, microtubule {dices[1]:.2f}, ribosome {dices[2]:.2f}")

🔍 **What just happened?** On a new tomogram of the *same kind* the networks work about as well as on the tomogram they
were trained from (some features a little better, some a little worse) — what was learned is the look of membranes,
microtubules and ribosomes, not one particular cell.
With noisier images and a different defocus (a different CTF), performance drops: the domain-shift problem of
Module 20. The paper saw the same pattern for ribosomes in cyanobacteria: the authors report about 89 % agreement on the
training tomogram, 86 % on another tomogram taken the same way (with a phase plate), and 78 % on a tomogram taken
without the phase plate.

---
## 7 · From annotation to particles

An annotation becomes biology when it is measured or averaged. The paper extracted microtubule segments and ribosomes
from the annotations and computed subtomogram averages (the authors report e.g. 759 ribosomes). The first step is to
turn the ribosome annotation into **positions**: group connected annotated voxels into blobs, keep blobs of a plausible
size (at least ~30 % of one ribosome's volume, computed from the template), and take their centres.

In [ ]:
# ── 7.1 Ribosome positions from the annotation ──
blobs, n_blobs = ndimage.label(merged == 3)
sizes = ndimage.sum(np.ones_like(blobs), blobs, index=np.arange(1, n_blobs + 1))
expected = (RIBO_TEMPLATE > 0.25).sum() / 4                               # one ribosome, in half-resolution voxels
keep = np.where(sizes >= 0.3 * expected)[0] + 1                            # ignore blobs much smaller than a ribosome
found = np.array(ndimage.center_of_mass(merged == 3, blobs, keep)) * [1, 2, 2]   # back to full-resolution voxels
res = match_points(found, ribo_centres, radius=8.0)
print(f"one ribosome ≈ {expected:.0f} half-resolution voxels; the largest blob ≈ {sizes.max() / expected:.1f} ribosomes")
print(f"(a) one position per blob    : {len(keep):3d} positions → precision {res['precision']:.2f}, recall {res['recall']:.2f}, "
      f"F1 {res['f1']:.2f} ({res['tp']} of {len(ribo_centres)} ribosomes found within 8 nm)")

# (b) smooth the ribosome annotation and keep its local maxima (each the highest point within ±6 voxels)
smooth = ndimage.gaussian_filter((merged == 3).astype(np.float32), 3.0)
peaks = np.argwhere((smooth == ndimage.maximum_filter(smooth, size=13)) & (smooth > 0.4)) * [1, 2, 2]
res_pk = match_points(peaks, ribo_centres, radius=8.0)
print(f"(b) peaks of the smoothed map: {len(peaks):3d} positions → precision {res_pk['precision']:.2f}, recall {res_pk['recall']:.2f}, "
      f"F1 {res_pk['f1']:.2f} ({res_pk['tp']} of {len(ribo_centres)} ribosomes found within 8 nm)")

🔍 **What just happened?** The obvious recipe (a) — one position per connected blob — fails in a crowded cell: touching
ribosomes merge into blobs worth several ribosomes, whose single centre lies between them, so most ribosomes are
missed. Recipe (b) — smooth the annotation and keep its local maxima at least about a ribosome radius apart — separates
touching neighbours and finds (nearly) every ribosome from the annotation alone, at the price of some false positives
(noise patches and fragments that were annotated as ribosome). Each position could now be boxed out and passed to the
alignment and averaging methods of Modules 12–14 — exactly the route from annotation to structure that the paper took.

✏️ **Your turn.** Increase the number of positive tiles for ribosomes to 20 (`pick_tiles(..., n_pos=20)`) and re-run
Sections 2–7. How much do recall and F1 improve?

### 📄 Paper connection — Chen *et al.* 2017: CNNs for automated annotation of cellular tomograms

**Paper:** Chen M, Dai W, Sun SY, Jonasch D, He CY, Schmid MF, Chiu W, Ludtke SJ. *Convolutional neural networks for
automated annotation of cellular cryo-electron tomograms.* Nature Methods 14:983–985 (2017).
[doi:10.1038/nmeth.4405](https://doi.org/10.1038/nmeth.4405) (preprint [arXiv:1701.05567](https://arxiv.org/abs/1701.05567))

- **Question it answers:** can tomograms be annotated automatically from a few minutes of manual labelling, instead of
  days?
- **What the paper actually does:** preprocesses tomograms (low-pass ≈ 20 Å, high-pass ≈ 4000 Å, normalise, clamp ±3 sd);
  trains one network per feature from ~10 hand-annotated positive 64 × 64 tiles and ~100 negative tiles (positives
  replicated in random orientations); the network has a 40 × 15 × 15 convolutional layer with ReLU, 2 × 2 max-pooling, a
  "fully connected" layer of 40 neurons with independent 15 × 15 kernels for each of the 1,600 connections, and one
  output neuron with activation $y=\min(1,x)$; the loss is the logarithm of the squared error against the 2×-shrunk
  annotation, with L1 weight decay; SGD with batches of 20 for 20 iterations, learning rate 0.01 decreasing 10 % per
  iteration — under 10 minutes per feature on a GTX 1080. Networks are applied slice by slice in 2D; outputs of several
  networks are normalised (mean 1 on annotated positives) and merged by taking the highest.
- **Result reported:** annotations of membranes, microtubules, ribosomes, mitochondria, vesicles, RuBisCO, carboxysomes,
  phages and thylakoids in tomograms of PC12 cells (7.2 Å/pixel), platelets (12.0 Å), cyanobacteria (phase plate,
  5.2 Å) and trypanosomes (5.3 Å); ribosome annotations reused across tomograms (≈ 89 % self, 86 % another phase-plate
  tomogram, 78 % a conventional one; 91.3 % with 10 positive / 100 negative tiles); subtomogram averages from annotations
  (microtubules N = 511, 677, 312; ribosomes N = 759). Processing a 4096 × 4096 × 1024 tomogram took ~20 h per feature
  unbinned, ~2.5 h binned by 2.
- **What our toy keeps:** preprocessing, tile-based training with replicated positives, the layer design (fewer channels),
  capped linear output, log-squared-error loss with L1, slice-by-slice application, normalised merging, reuse tests.
- **What our toy leaves out:** real hand-drawn annotations (ours are perfect), 40-channel networks, SGD schedule (we use
  Adam), more features and cell types.
- **Limitations stated:** unpredictable responses to untrained features (carbon edges, gold), a maximum scale of about
  30 pixels (centres of large vesicles missed), best performance on the same cell type and conditions.
- **Code:** EMAN2 (`e2tomoseg` tools), <https://github.com/cryoem/eman2>.

---
## ✅ Checkpoint

1. Why train one network per feature rather than one network for all features?
   <details><summary>Answer</summary>Each network answers one simple question from a handful of labels; adding a feature means labelling a few tiles and training one more small network. The merge step then resolves conflicts.</details>
2. Why are positive tiles replicated in random orientations?
   <details><summary>Answer</summary>To balance ~10 positives against ~100 negatives and to teach that a membrane or filament can run in any direction (orientation invariance).</details>
3. Why does the output map have half the size of the input, and what follows for the labels?
   <details><summary>Answer</summary>The 2 × 2 max-pooling halves the image; the annotation must be shrunk 2× to match.</details>
4. What do the large 15 × 15 kernels buy, and what do they limit?
   <details><summary>Answer</summary>Context: enough pixels to recognise double lines and tube walls. But the network still sees only ~30 pixels at once, so very large objects are recognised only by their edges.</details>
5. Why apply the networks slice by slice instead of in 3D?
   <details><summary>Answer</summary>Simpler, faster networks trained on 2D tiles; slices parallel to the specimen are the least affected by missing-wedge elongation.</details>
6. A gold bead is annotated as a microtubule. What would you do?
   <details><summary>Answer</summary>Train an extra network for the gold class so that the merge can assign those voxels to it (and/or mask beads before annotation).</details>

## 🧠 Questions & discussion

1. Our labels were perfect. Real hand-drawn labels are fuzzy and incomplete. How might that change the networks?
2. The paper's networks are tiny by today's standards. Compare them with the 3D U-Nets of Module 17 and the foundation
   models of Module 32: what did each generation add?
3. Why does clamping at ±3 sd matter for gold beads and carbon edges?
4. A network trained on PC12 cells is applied to plant cells. Which features would you trust first?
5. How would you combine automatic annotation with the template matching of Module 11 to find ribosomes?

## 🚀 Bonus — going further

- **A gold network.** Train a fourth network for `feature = 4` and repeat the merge. Do the false microtubules disappear?
- **Paper-sized networks.** Set `CHANNELS = 40` and use SGD (lr 0.01, −10 % per epoch, 20 epochs). How long does it take
  on your machine, and is it better?
- **3D context.** Feed three neighbouring slices as three input channels. Does the missing-wedge elongation along z change?
- **Real software:** EMAN2's tomogram segmentation tutorial walks through the same steps on real data
  (<https://blake.bcm.edu/emanwiki/EMAN2/Programs/tomoseg>).

## 📝 Lab note

```text
Module 18 — date:
Input: simulated 64 × 192 × 192 tomogram (1 nm voxels; vesicles, microtubules, ribosomes, gold), SNR 0.1, ±60°
Output: three feature networks, slice-by-slice feature maps, merged annotation, ribosome positions, reuse tests
Method: band-pass + clamp, tile-based training with replicated positives, 15×15 CNN, log squared error, normalised merge
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Chen et al. 2017 (Nat Methods)
```

## 📚 References

1. Chen M, Dai W, Sun SY, Jonasch D, He CY, Schmid MF, Chiu W, Ludtke SJ. Convolutional neural networks for automated annotation of cellular cryo-electron tomograms. *Nat Methods* 14:983–985 (2017). doi:10.1038/nmeth.4405
2. Tang G *et al.* EMAN2: an extensible image processing suite for electron microscopy. *J Struct Biol* 157:38–46 (2007). doi:10.1016/j.jsb.2006.05.009
3. Ronneberger O, Fischer P, Brox T. U-Net: convolutional networks for biomedical image segmentation. *MICCAI* (2015). arXiv:1505.04597
4. Moebel E *et al.* Deep learning improves macromolecule identification in 3D cellular cryo-electron tomograms. *Nat Methods* 18:1386–1394 (2021). doi:10.1038/s41592-021-01275-4

## ➡️ Next

**[19 · Learning with few labels: self-supervision, few-shot, active and open-set learning](19_learning_with_few_labels.ipynb)**
— ten tiles were enough here; Module 19 asks how far "few labels" can go for classifying particles.
Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).

In [ ]:
# ── Total run time ──
print(f"Notebook finished in {time.time() - T_START:.0f} s")